In [ ]:
from pathlib import Path
import polars as pl

# Define paths
DATA_DIR = Path("../home-credit-default-risk")
PROCESSED_DIR = DATA_DIR / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

bureau_path = DATA_DIR / "bureau.csv"
bureau_balance_path = DATA_DIR / "bureau_balance.csv"
output_path = PROCESSED_DIR / "aggregate_bureau.parquet"

print("Starting unified bureau & bureau_balance feature engineering pipeline...")

# Step 1: Aggregate bureau_balance at the loan level (SK_ID_BUREAU)
df_bb = pl.scan_csv(bureau_balance_path)
bb_agg = (
    df_bb
    .group_by("SK_ID_BUREAU")
    .agg([
        # Find the worst numeric delinquency status (0 to 5) for each loan
        pl.col("STATUS")
          .filter(pl.col("STATUS").is_in(["0", "1", "2", "3", "4", "5"]))
          .max()
          .alias("bb_max_delinquency"),
          
        # Count total on-time months ('0') for each loan
        (pl.col("STATUS") == "0").sum().alias("bb_count_ontime"),
          
        # Count total closed/paid status months ('C') for each loan
        (pl.col("STATUS") == "C").sum().alias("bb_count_closed"),
    ])
    .collect()  # Collect loan-level aggregations into memory
)

# Step 2: Read raw bureau and join the loan-level balance features 
# (This happens BEFORE grouping by SK_ID_CURR, so SK_ID_BUREAU is still available!)
df_bureau = pl.scan_csv(bureau_path)

df_bureau_joined = (
    df_bureau
    .join(bb_agg.lazy(), on="SK_ID_BUREAU", how="left")
)

# Step 3: Run a single unified group-by at the client level (SK_ID_CURR)
print("Rolling everything up to the client level (SK_ID_CURR)...")
bureau_client_agg = (
    df_bureau_joined
    .group_by("SK_ID_CURR")
    .agg([
        # --- Original Bureau Features ---
        pl.len().alias("bureau_loan_count"),
        (pl.col("CREDIT_ACTIVE") == "Active").sum().alias("bureau_active_count"),
        pl.col("AMT_CREDIT_SUM_DEBT").filter(pl.col("CREDIT_ACTIVE") == "Active").sum().alias("bureau_active_debt_sum"),
        pl.col("AMT_CREDIT_SUM").sum().alias("bureau_total_customer_credit_sum"),
        pl.col("AMT_CREDIT_SUM_DEBT").sum().alias("bureau_total_customer_debt_sum"),
        pl.col("AMT_CREDIT_MAX_OVERDUE").max().alias("bureau_max_amt_overdue"),
        pl.col("DAYS_CREDIT").max().alias("bureau_days_credit_max"),
        pl.col("DAYS_CREDIT").min().alias("bureau_days_credit_min"),
        
        # --- New Bureau Balance Features ---
        pl.col("bb_max_delinquency").max().alias("client_worst_delinquency_ever"),
        pl.col("bb_count_ontime").sum().alias("client_total_ontime_months"),
        pl.col("bb_count_closed").sum().alias("client_total_closed_accounts"),
    ])
    .collect()  # Pull the final client feature set into memory
)

# Save the final comprehensive feature set to Parquet
bureau_client_agg.write_parquet(output_path)

print(f"\nSuccess! Unified features created and saved to: {output_path}")
print(f"Final shape (Clients x Features): {bureau_client_agg.shape}")
print("\nPreview of your final client-level features:")
print(bureau_client_agg.head(5))

Starting unified bureau & bureau_balance feature engineering pipeline...
Rolling everything up to the client level (SK_ID_CURR)...

Success! Unified features created and saved to: ..\home-credit-default-risk\Processed\aggregate_bureau.parquet
Final shape (Clients x Features): (305811, 12)

Preview of your final client-level features:
shape: (5, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ SK_ID_CUR ┆ bureau_lo ┆ bureau_ac ┆ bureau_ac ┆ … ┆ bureau_da ┆ client_wo ┆ client_to ┆ client_t │
│ R         ┆ an_count  ┆ tive_coun ┆ tive_debt ┆   ┆ ys_credit ┆ rst_delin ┆ tal_ontim ┆ otal_clo │
│ ---       ┆ ---       ┆ t         ┆ _sum      ┆   ┆ _min      ┆ quency_ev ┆ e_months  ┆ sed_acco │
│ i64       ┆ u32       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ er        ┆ ---       ┆ unts     │
│           ┆           ┆ u32       ┆ f64       ┆   ┆ i64       ┆ ---       ┆ u32       ┆ ---      │
│           ┆           ┆           ┆      